# Preparar Dataset Extendido 2019-2025 para Reentrenamiento

**Entrada:** `master_dataset_extendido.csv` (7,707 filas provincia-mes, mezcla crudos 2019-2020 + z-scores 2021-2025)

**Pipeline:**
1. Agregar a nivel mensual (media por mes, igual que v6)
2. Incorporar NLP (nlp_index + lag) y flags (tiene_nlp, es_shock)
3. Re-normalizar con StandardScaler train/test por separado
4. Verificar integridad y guardar

**Salida:** `master_escalado_extendido.csv` (~80 filas mensuales, normalizado consistente)

In [1]:
import numpy as np
import pandas as pd
import json
from pathlib import Path
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

ROOT = Path.cwd()
while not (ROOT / 'CLAUDE.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

DATA_PATH = ROOT / 'data/processed/master_dataset_extendido.csv'
NLP_PATH  = ROOT / 'notebooks/fase2/output/01_nlp_sentimiento/sentimiento_mensual_v2.csv'
OUT_DIR   = ROOT / 'resultados'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raiz: {ROOT}")
print(f"Dataset: {DATA_PATH.exists()}")
print(f"NLP:     {NLP_PATH.exists()}")

Raiz: C:\Machine-learming\Machine-Learning-Multimodal--Agro-NLP-Clima-
Dataset: True
NLP:     True


---
## PASO 1 — Agregar a nivel mensual

Replica el pipeline de v6: `groupby('fecha_evento').mean(numeric_only=True)`

**Advertencia de escala:** Las filas 2019-2020 tienen valores crudos y las 2021-2025 z-scores. La agregacion mensual (media de provincias) preserva esta diferencia de escala. El PASO 2 la resuelve aplicando StandardScaler sobre todo el rango.

In [2]:
# PASO 1a: Cargar y agregar a nivel mensual
df_raw = pd.read_csv(DATA_PATH, parse_dates=['fecha_evento'])
print(f"Raw (provincia-mes): {df_raw.shape}")

df = df_raw.groupby('fecha_evento').mean(numeric_only=True).reset_index()
df = df.sort_values('fecha_evento').reset_index(drop=True)

print(f"Agregado mensual:    {df.shape}")
print(f"Rango: {df['fecha_evento'].min().date()} -> {df['fecha_evento'].max().date()}")
print(f"Meses totales: {len(df)}")
print(f"\nColumnas ({len(df.columns)}): {df.columns.tolist()}")

# Diagnostico de escala: comparar medias 2019-2020 vs 2021-2025
mask_1920 = df['fecha_evento'].dt.year.isin([2019, 2020])
mask_2125 = df['fecha_evento'].dt.year >= 2021

print(f"\n--- Diagnostico de escala (media por periodo) ---")
cols_check = ['produccion_t', 'precio_chacra_kg', 'T2M', 'PRECTOTCORR']
for c in cols_check:
    v1 = df.loc[mask_1920, c].mean()
    v2 = df.loc[mask_2125, c].mean()
    print(f"  {c:30s}  2019-20={v1:10.3f}  2021-25={v2:10.3f}")

print("\n(Los valores de 2019-20 son crudos, los de 2021-25 son z-scores ~ 0)")
print("El PASO 2 normaliza todo junto para resolver esta diferencia.")

Raw (provincia-mes): (7707, 24)
Agregado mensual:    (80, 22)
Rango: 2019-01-01 -> 2025-08-01
Meses totales: 80

Columnas (22): ['fecha_evento', 'produccion_t', 'precio_chacra_kg', 'num_emergencias', 'total_afectados', 'hectareas_cultivo_perdidas', 'ALLSKY_SFC_SW_DWN', 'PRECTOTCORR', 'QV2M', 'RH2M', 'T2M', 'T2M_MAX', 'T2M_MIN', 'WS2M', 'lat', 'lon', 'month_sin', 'month_cos', 'mes_num', 'trimestre_num', 'trimestre_sin', 'trimestre_cos']

--- Diagnostico de escala (media por periodo) ---
  produccion_t                    2019-20=   332.160  2021-25=     0.000
  precio_chacra_kg                2019-20=     1.201  2021-25=     0.000
  T2M                             2019-20=    19.290  2021-25=     0.000
  PRECTOTCORR                     2019-20=     2.171  2021-25=    -0.000

(Los valores de 2019-20 son crudos, los de 2021-25 son z-scores ~ 0)
El PASO 2 normaliza todo junto para resolver esta diferencia.


In [3]:
# PASO 1b: Incorporar NLP sentiment (igual que v6 cell 3-5)
df_nlp = pd.read_csv(NLP_PATH, encoding='utf-8-sig')
fc = [c for c in df_nlp.columns if any(k in c.lower() for k in ['fecha', 'periodo', 'mes', 'month'])][0]
df_nlp = df_nlp.rename(columns={fc: 'fecha_evento'})
df_nlp['fecha_evento'] = pd.to_datetime(df_nlp['fecha_evento'])
df_nlp = df_nlp.sort_values('fecha_evento').reset_index(drop=True)

# nlp_index = avg_sentiment * log1p(n_noticias_beto)  (formula v6)
df_nlp['nlp_index']      = df_nlp['avg_sentiment'] * np.log1p(df_nlp['n_noticias_beto'])
df_nlp['nlp_index_lag1'] = df_nlp['nlp_index'].shift(1).fillna(0)

print(f"NLP sentiment: {df_nlp.shape}")
print(f"Rango NLP: {df_nlp['fecha_evento'].min().date()} -> {df_nlp['fecha_evento'].max().date()}")

# Merge NLP con dataset mensual (left join — 2019-2020 quedan como NaN -> 0)
df = df.merge(df_nlp[['fecha_evento', 'nlp_index', 'nlp_index_lag1']], on='fecha_evento', how='left')
df['nlp_index']      = df['nlp_index'].fillna(0)
df['nlp_index_lag1'] = df['nlp_index_lag1'].fillna(0)

# Agregar flag tiene_nlp
df['tiene_nlp'] = (df['fecha_evento'].dt.year >= 2021).astype(int)

# Agregar es_shock: variacion mensual de produccion_t > 20%
df['_pct_change'] = df['produccion_t'].pct_change().abs()
df['es_shock'] = (df['_pct_change'] > 0.20).astype(int)
df.loc[0, 'es_shock'] = 0  # primer mes sin referencia
df = df.drop(columns='_pct_change')

# Verificar 2019-2020
mask_1920 = df['fecha_evento'].dt.year.isin([2019, 2020])
print(f"\n--- Verificacion 2019-2020 ---")
print(f"  tiene_nlp:  {df.loc[mask_1920, 'tiene_nlp'].unique()}  (esperado: [0])")
print(f"  nlp_index:  media={df.loc[mask_1920, 'nlp_index'].mean():.4f}  (esperado: 0)")
print(f"  nlp_index_lag1: media={df.loc[mask_1920, 'nlp_index_lag1'].mean():.4f}  (esperado: 0)")

print(f"\nDataset final pre-normalizacion: {df.shape}")
print(f"Columnas ({len(df.columns)}): {df.columns.tolist()}")
df.head()

NLP sentiment: (60, 8)
Rango NLP: 2021-01-01 -> 2025-12-01

--- Verificacion 2019-2020 ---
  tiene_nlp:  [0]  (esperado: [0])
  nlp_index:  media=0.0000  (esperado: 0)
  nlp_index_lag1: media=0.0000  (esperado: 0)

Dataset final pre-normalizacion: (80, 26)
Columnas (26): ['fecha_evento', 'produccion_t', 'precio_chacra_kg', 'num_emergencias', 'total_afectados', 'hectareas_cultivo_perdidas', 'ALLSKY_SFC_SW_DWN', 'PRECTOTCORR', 'QV2M', 'RH2M', 'T2M', 'T2M_MAX', 'T2M_MIN', 'WS2M', 'lat', 'lon', 'month_sin', 'month_cos', 'mes_num', 'trimestre_num', 'trimestre_sin', 'trimestre_cos', 'nlp_index', 'nlp_index_lag1', 'tiene_nlp', 'es_shock']


,fecha_evento,produccion_t,precio_chacra_kg,num_emergencias,total_afectados,hectareas_cultivo_perdidas,ALLSKY_SFC_SW_DWN,PRECTOTCORR,QV2M,RH2M,...,month_sin,month_cos,mes_num,trimestre_num,trimestre_sin,trimestre_cos,nlp_index,nlp_index_lag1,tiene_nlp,es_shock
0,2019-01-01,402.787246,0.995238,1.637681,74.333333,1.259130,16.900870,3.516812,13.508116,79.916957,...,0.500000,8.660254e-01,1.0,1.0,1.000000e+00,6.123234e-17,0.0,0.0,0,0
1,2019-02-01,433.112500,0.986471,7.236111,476.638889,118.517639,15.884306,3.519861,14.095278,82.232639,...,0.866025,5.000000e-01,2.0,1.0,1.000000e+00,6.123234e-17,0.0,0.0,0,0
2,2019-03-01,421.733500,1.015640,7.812500,138.900000,6.735500,16.584250,3.412375,13.480375,81.771625,...,1.000000,6.123234e-17,3.0,1.0,1.000000e+00,6.123234e-17,0.0,0.0,0,0
3,2019-04-01,363.863977,0.993889,4.056818,50.079545,1.642955,17.355341,2.196705,12.786136,78.705682,...,0.866025,-5.000000e-01,4.0,2.0,1.224647e-16,-1.000000e+00,0.0,0.0,0,0
4,2019-05-01,345.860000,1.033620,1.434783,28.554348,8.018261,17.144130,1.353478,11.833696,75.413804,...,0.500000,-8.660254e-01,5.0,2.0,1.224647e-16,-1.000000e+00,0.0,0.0,0,0


---
## PASO 2 — Re-normalizar con StandardScaler

- **Train:** desde 2019-01 hasta el mes n_total - 12 (inclusive)
- **Test:** ultimos 12 meses
- `StandardScaler.fit()` solo sobre train, luego `.transform()` ambos splits
- Columnas excluidas de normalizacion: `fecha_evento`, `tiene_nlp`, `es_shock`, `produccion_t` (target)
- Guardar parametros del scaler en `resultados/scaler_extendido.json`

In [4]:
# PASO 2a: Split cronologico
n_total = len(df)
n_test  = 12
n_train = n_total - n_test

df_train = df.iloc[:n_train].copy()
df_test  = df.iloc[n_train:].copy()

print(f"n_total: {n_total}")
print(f"n_train: {n_train}  | {df_train['fecha_evento'].min().date()} -> {df_train['fecha_evento'].max().date()}")
print(f"n_test:  {n_test}  | {df_test['fecha_evento'].min().date()} -> {df_test['fecha_evento'].max().date()}")

# Confirmar que 2019-2020 (24 meses) estan completamente en train
n_1920_train = df_train['fecha_evento'].dt.year.isin([2019, 2020]).sum()
print(f"\nMeses 2019-2020 en train: {n_1920_train}  (esperado: ~24)")

n_total: 80
n_train: 68  | 2019-01-01 -> 2024-08-01
n_test:  12  | 2024-09-01 -> 2025-08-01

Meses 2019-2020 en train: 24  (esperado: ~24)


In [5]:
# PASO 2b: Definir columnas a escalar vs excluir
TARGET     = 'produccion_t'
NO_SCALE   = ['fecha_evento', 'tiene_nlp', 'es_shock', TARGET]
COLS_SCALE = [c for c in df.columns if c not in NO_SCALE]

print(f"Target:              {TARGET}")
print(f"Excluidas ({len(NO_SCALE)}):      {NO_SCALE}")
print(f"A escalar ({len(COLS_SCALE)}):  {COLS_SCALE}")

# Fit scaler SOLO sobre train
scaler = StandardScaler()
scaler.fit(df_train[COLS_SCALE])

# Transformar train y test
df_train_scaled = df_train.copy()
df_test_scaled  = df_test.copy()

df_train_scaled[COLS_SCALE] = scaler.transform(df_train[COLS_SCALE])
df_test_scaled[COLS_SCALE]  = scaler.transform(df_test[COLS_SCALE])

# Escalar target por separado (scaler independiente, como en v6)
scaler_y = StandardScaler()
scaler_y.fit(df_train[[TARGET]])
df_train_scaled[TARGET] = scaler_y.transform(df_train[[TARGET]])
df_test_scaled[TARGET]  = scaler_y.transform(df_test[[TARGET]])

print(f"\nEscalado completado.")
print(f"Train escalado: {df_train_scaled.shape}")
print(f"Test escalado:  {df_test_scaled.shape}")

Target:              produccion_t
Excluidas (4):      ['fecha_evento', 'tiene_nlp', 'es_shock', 'produccion_t']
A escalar (22):  ['precio_chacra_kg', 'num_emergencias', 'total_afectados', 'hectareas_cultivo_perdidas', 'ALLSKY_SFC_SW_DWN', 'PRECTOTCORR', 'QV2M', 'RH2M', 'T2M', 'T2M_MAX', 'T2M_MIN', 'WS2M', 'lat', 'lon', 'month_sin', 'month_cos', 'mes_num', 'trimestre_num', 'trimestre_sin', 'trimestre_cos', 'nlp_index', 'nlp_index_lag1']

Escalado completado.
Train escalado: (68, 26)
Test escalado:  (12, 26)


In [6]:
# PASO 2c: Guardar parametros del scaler
scaler_params = {
    'features': COLS_SCALE,
    'mean': {col: float(m) for col, m in zip(COLS_SCALE, scaler.mean_)},
    'scale': {col: float(s) for col, s in zip(COLS_SCALE, scaler.scale_)},
    'target': TARGET,
    'target_mean': float(scaler_y.mean_[0]),
    'target_scale': float(scaler_y.scale_[0]),
    'n_train': n_train,
    'n_test': n_test,
    'train_range': f"{df_train['fecha_evento'].min().date()} -> {df_train['fecha_evento'].max().date()}",
    'test_range': f"{df_test['fecha_evento'].min().date()} -> {df_test['fecha_evento'].max().date()}",
}

scaler_path = OUT_DIR / 'scaler_extendido.json'
with open(scaler_path, 'w') as f:
    json.dump(scaler_params, f, indent=2)

print(f"Scaler guardado en: {scaler_path}")
print(f"\nParametros del scaler (features):")
for col in COLS_SCALE:
    print(f"  {col:30s}  mean={scaler_params['mean'][col]:12.4f}  scale={scaler_params['scale'][col]:12.4f}")
print(f"\nTarget ({TARGET}):")
print(f"  mean={scaler_params['target_mean']:.4f}  scale={scaler_params['target_scale']:.4f}")

Scaler guardado en: C:\Machine-learming\Machine-Learning-Multimodal--Agro-NLP-Clima-\resultados\scaler_extendido.json

Parametros del scaler (features):
  precio_chacra_kg                mean=      0.4210  scale=      0.7012
  num_emergencias                 mean=      0.9489  scale=      1.6464
  total_afectados                 mean=     26.8441  scale=     67.9846
  hectareas_cultivo_perdidas      mean=      4.3576  scale=     15.3152
  ALLSKY_SFC_SW_DWN               mean=      6.2301  scale=      8.4103
  PRECTOTCORR                     mean=      0.7438  scale=      1.3042
  QV2M                            mean=      4.3369  scale=      5.9429
  RH2M                            mean=     26.2652  scale=     35.7589
  T2M                             mean=      6.8100  scale=      9.2434
  T2M_MAX                         mean=      9.7428  scale=     13.1955
  T2M_MIN                         mean=      4.5372  scale=      6.2564
  WS2M                            mean=      0.5299  sc

---
## PASO 3 — Verificar integridad

- Confirmar n_train, n_test, rango temporal
- Confirmar que los 24 primeros meses (2019-2020) estan en train
- Confirmar que nlp_index tiene media ~0 en train tras escalar
- Mostrar 5 primeras y 5 ultimas filas del dataset escalado

In [7]:
# PASO 3a: Verificaciones de integridad
dataset_scaled = pd.concat([df_train_scaled, df_test_scaled], ignore_index=True)

print("="*70)
print("VERIFICACIONES DE INTEGRIDAD")
print("="*70)

# 1. Tamanos y rangos
print(f"\n[1] Tamanos:")
print(f"    n_train = {n_train}")
print(f"    n_test  = {n_test}")
print(f"    n_total = {len(dataset_scaled)}")

print(f"\n[2] Rangos temporales:")
print(f"    Train: {df_train_scaled['fecha_evento'].min().date()} -> {df_train_scaled['fecha_evento'].max().date()}")
print(f"    Test:  {df_test_scaled['fecha_evento'].min().date()} -> {df_test_scaled['fecha_evento'].max().date()}")

# 2. Primeros 24 meses en train
meses_1920 = df_train_scaled[df_train_scaled['fecha_evento'].dt.year.isin([2019, 2020])]
print(f"\n[3] Meses 2019-2020 en train: {len(meses_1920)} (esperado: ~24)")
assert len(meses_1920) >= 20, "ERROR: faltan meses 2019-2020 en train"
print("    OK - todos los meses 2019-2020 estan en train")

# 3. nlp_index media ~0 en train
nlp_mean_train = df_train_scaled['nlp_index'].mean()
print(f"\n[4] nlp_index media en train (escalado): {nlp_mean_train:.6f}")
print(f"    (esperado: ~0 por StandardScaler)")

# 4. tiene_nlp y es_shock NO fueron tocados
print(f"\n[5] tiene_nlp valores unicos: {sorted(dataset_scaled['tiene_nlp'].unique())}")
print(f"    es_shock valores unicos:  {sorted(dataset_scaled['es_shock'].unique())}")
print(f"    tiene_nlp en 2019-2020:   {meses_1920['tiene_nlp'].unique()} (esperado: [0])")

# 5. Nulos
nulos = dataset_scaled.isnull().sum()
print(f"\n[6] Nulos por columna:")
if nulos.any():
    print(nulos[nulos > 0])
else:
    print("    Ninguno")

VERIFICACIONES DE INTEGRIDAD

[1] Tamanos:
    n_train = 68
    n_test  = 12
    n_total = 80

[2] Rangos temporales:
    Train: 2019-01-01 -> 2024-08-01
    Test:  2024-09-01 -> 2025-08-01

[3] Meses 2019-2020 en train: 24 (esperado: ~24)
    OK - todos los meses 2019-2020 estan en train

[4] nlp_index media en train (escalado): 0.000000
    (esperado: ~0 por StandardScaler)

[5] tiene_nlp valores unicos: [np.int64(0), np.int64(1)]
    es_shock valores unicos:  [np.int64(0), np.int64(1)]
    tiene_nlp en 2019-2020:   [0] (esperado: [0])

[6] Nulos por columna:
    Ninguno


In [8]:
# PASO 3b: Mostrar primeras 5 y ultimas 5 filas
print("--- 5 PRIMERAS FILAS (2019) ---")
display(dataset_scaled.head(5))

print("\n--- 5 ULTIMAS FILAS (2025) ---")
display(dataset_scaled.tail(5))

--- 5 PRIMERAS FILAS (2019) ---


,fecha_evento,produccion_t,precio_chacra_kg,num_emergencias,total_afectados,hectareas_cultivo_perdidas,ALLSKY_SFC_SW_DWN,PRECTOTCORR,QV2M,RH2M,...,month_sin,month_cos,mes_num,trimestre_num,trimestre_sin,trimestre_cos,nlp_index,nlp_index_lag1,tiene_nlp,es_shock
0,2019-01-01,1.720446,0.818900,0.418354,0.698529,-0.202314,1.268786,2.126223,1.543211,1.500376,...,0.658698,1.275497,-1.549276,-1.294024,1.373655,0.063457,0.304853,0.275792,0,0
1,2019-02-01,1.903160,0.806398,3.818678,6.616125,7.454028,1.147914,2.128561,1.642011,1.565134,...,1.176964,0.757231,-1.255000,-1.294024,1.373655,0.063457,0.304853,0.275792,0,0
2,2019-03-01,1.834600,0.847995,4.168760,1.648254,0.155263,1.231139,2.046146,1.538543,1.552242,...,1.366662,0.049267,-0.960724,-1.294024,1.373655,0.063457,0.304853,0.275792,0,0
3,2019-04-01,1.485928,0.816977,1.887667,0.341775,-0.177252,1.322824,1.114022,1.421725,1.466502,...,1.176964,-0.658698,-0.666448,-0.386873,-0.020502,-1.374911,0.304853,0.275792,0,0
4,2019-05-01,1.377452,0.873638,0.295119,0.025157,0.239020,1.297710,0.467472,1.261461,1.374445,...,0.658698,-1.176964,-0.372172,-0.386873,-0.020502,-1.374911,0.304853,0.275792,0,0



--- 5 ULTIMAS FILAS (2025) ---


,fecha_evento,produccion_t,precio_chacra_kg,num_emergencias,total_afectados,hectareas_cultivo_perdidas,ALLSKY_SFC_SW_DWN,PRECTOTCORR,QV2M,RH2M,...,month_sin,month_cos,mes_num,trimestre_num,trimestre_sin,trimestre_cos,nlp_index,nlp_index_lag1,tiene_nlp,es_shock
75,2025-04-01,-0.706792,-0.658244,-0.728031,-0.396184,-0.285439,-0.770738,-0.470516,-0.677795,-0.717334,...,1.176964,-0.658698,-0.666448,-0.386873,-0.020502,-1.374911,0.217506,1.061501,1,1
76,2025-05-01,-0.706919,-0.725022,-0.728031,-0.396184,-0.285439,-0.800022,-0.655326,-0.709004,-0.722991,...,0.658698,-1.176964,-0.372172,-0.386873,-0.020502,-1.374911,0.679231,0.183074,1,1
77,2025-06-01,-0.707072,-0.728525,-0.728031,-0.396184,-0.285439,-0.849284,-0.073901,-0.728407,-0.720006,...,-0.049267,-1.366662,-0.077897,-0.386873,-0.020502,-1.374911,0.988349,0.673188,1,1
78,2025-07-01,-0.707015,-0.609094,-0.728031,-0.396184,-0.285439,-0.757443,-0.945432,-0.788943,-0.736324,...,-0.757231,-1.176964,0.216379,0.520278,-1.414659,0.063457,1.100537,1.001313,1,0
79,2025-08-01,-0.707074,-0.221150,-0.728031,-0.396184,-0.285439,-0.727076,-0.892061,-0.782474,-0.743283,...,-1.275497,-0.658698,0.510655,0.520278,-1.414659,0.063457,0.660261,1.120398,1,0


In [9]:
# PASO 3c: Guardar dataset escalado completo
output_path = ROOT / 'data/processed/master_escalado_extendido.csv'
dataset_scaled.to_csv(output_path, index=False)

print(f"Guardado en: {output_path}")
print(f"Tamano: {output_path.stat().st_size / 1024:.1f} KB")

# Resumen final
print(f"\n{'='*70}")
print(f"RESUMEN FINAL")
print(f"{'='*70}")
print(f"Archivo:     {output_path.name}")
print(f"Filas:       {len(dataset_scaled)} ({n_train} train + {n_test} test)")
print(f"Columnas:    {len(dataset_scaled.columns)}")
print(f"Rango:       {dataset_scaled['fecha_evento'].min().date()} -> {dataset_scaled['fecha_evento'].max().date()}")
print(f"Features escaladas:    {len(COLS_SCALE)}")
print(f"Target:                {TARGET} (escalado aparte)")
print(f"Flags sin escalar:     tiene_nlp, es_shock")
print(f"Scaler en:             {scaler_path.name}")
print(f"Nulos:                 0")
print(f"{'='*70}")

Guardado en: C:\Machine-learming\Machine-Learning-Multimodal--Agro-NLP-Clima-\data\processed\master_escalado_extendido.csv
Tamano: 36.8 KB

RESUMEN FINAL
Archivo:     master_escalado_extendido.csv
Filas:       80 (68 train + 12 test)
Columnas:    26
Rango:       2019-01-01 -> 2025-08-01
Features escaladas:    22
Target:                produccion_t (escalado aparte)
Flags sin escalar:     tiene_nlp, es_shock
Scaler en:             scaler_extendido.json
Nulos:                 0
